# Getting started

In [ ]:
#!pip install neuralforecast

### Import libraries

In [ ]:
import neuralforecast as nf
import numpy as np
import pandas as pd
import pytorch_lightning as pl
import torch as t

### Dataset

In [ ]:
Y_df, X_df, _ = nf.data.datasets.long_horizon.LongHorizon.load('data', 'ILI')
Y_df['ds'] = pd.to_datetime(Y_df['ds'])
X_df['ds'] = pd.to_datetime(X_df['ds'])
Y_df.head()

### Split train/test sets

In [ ]:
output_size = 24
Y_df_test = Y_df.groupby('unique_id').tail(output_size)
Y_df_train = Y_df.drop(Y_df_test.index)

X_df_train = X_df.drop(Y_df_test.index)

### Define WindowsDataset and TimeSeriesLoader

In [ ]:
input_size = 5 * output_size

In [ ]:
train_mask_df, val_mask_df, _ = nf.experiments.utils.get_mask_dfs(
    Y_df=Y_df_train,
    ds_in_val=7 * output_size,
    ds_in_test=0
)

In [ ]:
plot_df = Y_df_train.merge(
    train_mask_df.drop('available_mask', axis=1).rename(columns={'sample_mask': 'sample_mask_train'}),
    how='left',
    on=['unique_id', 'ds']
).merge(
    val_mask_df.drop('available_mask', axis=1).rename(columns={'sample_mask': 'sample_mask_val'}),
    how='left',
    on=['unique_id', 'ds']
)

In [ ]:
plot_df['y_train'] = np.where(plot_df['sample_mask_train'] == 1, plot_df['y'], np.nan)
plot_df['y_val'] = np.where(plot_df['sample_mask_val'] == 1, plot_df['y'], np.nan)

In [ ]:
plot_df.query('unique_id == "OT"').set_index('ds')[['y_train', 'y_val']].plot()

In [ ]:
train_dataset = nf.data.tsdataset.WindowsDataset(
    Y_df=Y_df_train, 
    X_df=X_df_train,
    f_cols=[f'ex_{i}' for i in range(1, 5)],
    input_size=input_size,
    output_size=output_size,
    mask_df=train_mask_df
)

In [ ]:
val_dataset = nf.data.tsdataset.WindowsDataset(
    Y_df=Y_df_train, 
    X_df=X_df_train,
    f_cols=[f'ex_{i}' for i in range(1, 5)],
    input_size=input_size,
    output_size=output_size,
    mask_df=val_mask_df
)

In [ ]:
train_loader = nf.data.tsloader.TimeSeriesLoader(
    train_dataset, batch_size=32, 
    n_windows=256,
    shuffle=True
)

In [ ]:
val_loader = nf.data.tsloader.TimeSeriesLoader(
    val_dataset, 
    batch_size=1
)

### Define model

In [ ]:
model = nf.models.nbeats.nbeats.NBEATS(
    n_time_in=input_size, n_time_out=output_size,
    n_x=4, n_x_hidden=[4],
    frequency='W-TUE', seasonality=4
)

### Train model with early stopping

In [ ]:
early_stopping = pl.callbacks.EarlyStopping(monitor="val_loss")

trainer = pl.Trainer(max_epochs=20,
                     gpus=-1 if t.cuda.is_available() else 0,
                     callbacks=[early_stopping])

trainer.fit(model, train_loader, val_loader)

In [ ]:
Y_df_forecast = model.forecast(Y_df_train, X_df=X_df)
Y_df_forecast.rename(columns={'y': 'y_hat'}, inplace=True)
Y_df_forecast.head()

In [ ]:
Y_df_plot = Y_df_test.merge(Y_df_forecast, how='left', on=['unique_id', 'ds'])
Y_df_plot.query('unique_id == "OT"').set_index('ds').plot()